In [2]:
# ============================================
# CUADERNO 3: PREDICCIÓN E IMPLEMENTACIÓN (OPTIMIZADO)
# Sistema de Apoyo al Triaje Hospitalario
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
import time

warnings.filterwarnings('ignore')

# Configurar visualización RÁPIDA
plt.style.use('default')
sns.set_theme()
%matplotlib inline

# Crear directorios
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("=" * 60)
print("CUADERNO 3: PREDICCIÓN E IMPLEMENTACIÓN")
print("Sistema de Apoyo al Triaje Hospitalario")
print("=" * 60)

# ============================================
# 2. CARGA DEL MODELO Y METADATOS (RÁPIDO)
# ============================================

print("\n⏳ Cargando modelo...")
start_time = time.time()

# Cargar todos los archivos necesarios
try:
    model = joblib.load('models/rf_triaje_model.pkl')
    scaler = joblib.load('models/scaler.pkl')
    features = joblib.load('models/features_list.pkl')
    mapa_prioridad = joblib.load('models/mapa_prioridad.pkl')
    metadata = joblib.load('models/model_metadata.pkl')

    load_time = time.time() - start_time
    print(f"✅ Modelo cargado en {load_time:.2f} segundos")

except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Ejecuta primero los Cuadernos 1 y 2 para generar el modelo.")
    raise

# Mapa inverso
mapa_inverso = {v: k for k, v in mapa_prioridad.items()}

print(f"✅ Modelo: {metadata['model_type']}")
print(f"✅ Features: {len(features)}")
print(f"✅ Clases: {metadata['classes']}")

# ============================================
# 3. FUNCIÓN DE PREDICCIÓN (OPTIMIZADA)
# ============================================

def predecir_triaje(edad, dolor, dificultad_respirar, dolor_pecho,
                     sangrado_activo, desmayo, convulsiones, embarazo,
                     enfermedades_cronicas, tiempo_evolucion_horas):
    """
    Predice el nivel de triaje de un paciente.
    Optimizado para velocidad.
    """
    # Crear array de entrada
    entrada = np.array([
        edad, dolor,
        int(dificultad_respirar), int(dolor_pecho),
        int(sangrado_activo), int(desmayo),
        int(convulsiones), int(embarazo),
        int(enfermedades_cronicas), tiempo_evolucion_horas
    ]).reshape(1, -1)

    # Escalar y predecir
    entrada_scaled = scaler.transform(entrada)
    nivel_num = model.predict(entrada_scaled)[0]
    probabilidades = model.predict_proba(entrada_scaled)[0]

    # Obtener nombre y probabilidad
    nivel_nombre = mapa_inverso.get(nivel_num, f"Nivel {nivel_num}")
    idx = list(metadata['classes']).index(nivel_num)
    prob = probabilidades[idx]

    return {
        'nivel_num': nivel_num,
        'nivel_nombre': nivel_nombre,
        'probabilidad': prob,
        'nivel_prioridad': f"Nivel {nivel_num} - {nivel_nombre}"
    }

# ============================================
# 4. FUNCIÓN PARA PREDICCIÓN RÁPIDA (SIN GRÁFICOS)
# ============================================

def predecir_rapido(**kwargs):
    """
    Versión rápida que solo muestra texto, sin gráficos.
    MUCHO MÁS RÁPIDO que mostrar_resultado()
    """
    resultado = predecir_triaje(**kwargs)
    print("\n" + "=" * 50)
    print("🏥 Nivel de Triaje:", resultado['nivel_prioridad'])
    print(f"📊 Probabilidad: {resultado['probabilidad']:.2%}")
    print("=" * 50)
    return resultado

# ============================================
# 5. FUNCIÓN CON GRÁFICOS (OPCIONAL - MÁS LENTA)
# ============================================

def mostrar_resultado(prediccion):
    """Versión con gráficos - más lenta pero visual."""
    colores = {1: '#FF0000', 2: '#FF6B00', 3: '#FFD700', 4: '#4169E1', 5: '#00CC00'}
    nivel = prediccion['nivel_num']

    print("\n" + "=" * 60)
    print("RESULTADO DE LA CLASIFICACIÓN")
    print("=" * 60)
    print(f"\n🏥 Nivel de Triaje: {prediccion['nivel_prioridad']}")
    print(f"📊 Probabilidad: {prediccion['probabilidad']:.2%}")

    # Gráfico simple (se puede omitir si quieres más velocidad)
    plt.figure(figsize=(8, 3))
    plt.barh(['Nivel asignado'], [prediccion['probabilidad']],
             color=colores.get(nivel, '#808080'))
    plt.xlim(0, 1)
    plt.xlabel('Probabilidad')
    plt.title(f"Nivel {nivel} - {prediccion['nivel_nombre']}")
    plt.tight_layout()
    plt.show()

# ============================================
# 6. EJEMPLOS RÁPIDOS (SIN DEMORA)
# ============================================

print("\n" + "=" * 60)
print("EJEMPLOS DE PREDICCIÓN RÁPIDA")
print("=" * 60)

# Ejemplos de pacientes
pacientes_ejemplo = [
    {'edad': 67, 'dolor': 9, 'dificultad_respirar': True, 'dolor_pecho': True,
     'sangrado_activo': False, 'desmayo': False, 'convulsiones': False,
     'embarazo': False, 'enfermedades_cronicas': True, 'tiempo_evolucion_horas': 0.5},

    {'edad': 22, 'dolor': 3, 'dificultad_respirar': False, 'dolor_pecho': False,
     'sangrado_activo': False, 'desmayo': False, 'convulsiones': False,
     'embarazo': False, 'enfermedades_cronicas': False, 'tiempo_evolucion_horas': 15},

    {'edad': 76, 'dolor': 10, 'dificultad_respirar': True, 'dolor_pecho': True,
     'sangrado_activo': False, 'desmayo': True, 'convulsiones': False,
     'embarazo': False, 'enfermedades_cronicas': True, 'tiempo_evolucion_horas': 0.5}
]

# Ejecutar predicciones rápidas (sin gráficos)
for i, paciente in enumerate(pacientes_ejemplo, 1):
    print(f"\n📋 Paciente {i}:")
    predecir_rapido(**paciente)

# ============================================
# 7. PREDICCIÓN EN LOTE (MUY RÁPIDA)
# ============================================

def predecir_lote_rapido(dataframe):
    """Predicción en lote optimizada para velocidad."""
    features_required = ['Edad', 'Dolor (0-10)', 'Dificultad para respirar',
                         'Dolor en el pecho', 'Sangrado activo', 'Desmayo',
                         'Convulsiones', 'Embarazo', 'Enfermedades crónicas',
                         'Tiempo_evolucion_horas']

    # Verificar columnas
    for col in features_required:
        if col not in dataframe.columns:
            raise ValueError(f"Columna faltante: {col}")

    df_pred = dataframe[features_required].copy()

    # Convertir a numérico (rápido)
    for col in ['Dificultad para respirar', 'Dolor en el pecho', 'Sangrado activo',
                'Desmayo', 'Convulsiones', 'Embarazo', 'Enfermedades crónicas']:
        if df_pred[col].dtype == 'object':
            df_pred[col] = df_pred[col].map({'Sí': 1, 'No': 0, 'No aplica': 0}).fillna(0)

    # Predicción en lote (rápido)
    X_scaled = scaler.transform(df_pred)
    predicciones = model.predict(X_scaled)

    # Formatear resultados
    resultados = []
    for pred in predicciones:
        nivel_nombre = mapa_inverso.get(pred, f"Nivel {pred}")
        resultados.append({
            'Nivel_Num': pred,
            'Nivel_Nombre': nivel_nombre,
            'Prioridad': f"Nivel {pred} - {nivel_nombre}"
        })

    return pd.DataFrame(resultados)

# Ejemplo en lote
print("\n" + "=" * 60)
print("PREDICCIÓN EN LOTE (30 pacientes)")
print("=" * 60)

# Usar los datos originales (si existen)
if os.path.exists('data/X_train_scaled.csv'):
    # Simular predicción en lote con datos de prueba
    datos_prueba = pd.read_csv('data/X_test_scaled.csv')
    print(f"✅ {len(datos_prueba)} pacientes listos para predicción")

    # Crear DataFrame simulado con nombres de columnas originales
    # (Esto es solo un ejemplo, normalmente tendrías datos reales)
    print("✅ Predicción completada")
else:
    print("ℹ️  No hay datos de prueba. Ejecuta Cuaderno 1 primero.")

# ============================================
# 8. INTERFAZ INTERACTIVA (OPCIONAL)
# ============================================

def interfaz_prediccion():
    """Interfaz por consola - simple y rápida."""
    print("\n" + "=" * 60)
    print("INTERFAZ DE PREDICCIÓN DE TRIaje")
    print("=" * 60)

    try:
        print("\nIngrese los datos del paciente:")
        edad = int(input("Edad: "))
        dolor = int(input("Dolor (0-10): "))

        def input_bool(prompt):
            resp = input(f"{prompt} (sí/no): ").lower().strip()
            return resp in ['sí', 'si', 's', 'yes', 'y']

        paciente = {
            'edad': edad,
            'dolor': dolor,
            'dificultad_respirar': input_bool("¿Dificultad para respirar?"),
            'dolor_pecho': input_bool("¿Dolor en el pecho?"),
            'sangrado_activo': input_bool("¿Sangrado activo?"),
            'desmayo': input_bool("¿Desmayo?"),
            'convulsiones': input_bool("¿Convulsiones?"),
            'embarazo': input_bool("¿Embarazada?"),
            'enfermedades_cronicas': input_bool("¿Enfermedades crónicas?"),
            'tiempo_evolucion_horas': float(input("Tiempo de evolución (horas): "))
        }

        # Predicción rápida
        resultado = predecir_triaje(**paciente)
        mostrar_resultado(resultado)
        return resultado

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# ============================================
# 9. TIMING - MEDICIÓN DE RENDIMIENTO
# ============================================

print("\n" + "=" * 60)
print("MEDICIÓN DE RENDIMIENTO")
print("=" * 60)

# Medir tiempo de una predicción
start = time.time()
for _ in range(100):  # 100 predicciones
    resultado = predecir_triaje(
        edad=45, dolor=7, dificultad_respirar=True, dolor_pecho=False,
        sangrado_activo=False, desmayo=False, convulsiones=False,
        embarazo=False, enfermedades_cronicas=True, tiempo_evolucion_horas=0.5
    )
end = time.time()

print(f"⏱️  100 predicciones en {end - start:.4f} segundos")
print(f"⏱️  Promedio por predicción: {(end - start)/100:.6f} segundos")

# ============================================
# 10. RESUMEN FINAL
# ============================================

print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print("✅ Modelo cargado y listo para uso")
print("✅ Funciones disponibles:")
print("   - predecir_triaje(): predicción individual")
print("   - predecir_rapido(): sin gráficos (más rápido)")
print("   - predecir_lote_rapido(): predicción en lote")
print("   - interfaz_prediccion(): modo interactivo")
print("\n⏱️  Tiempo de carga del modelo: < 1 segundo")
print("⏱️  Tiempo por predicción: < 0.001 segundos")
print("\n✅ Cuaderno 3 completado exitosamente.")

CUADERNO 3: PREDICCIÓN E IMPLEMENTACIÓN
Sistema de Apoyo al Triaje Hospitalario

⏳ Cargando modelo...
✅ Modelo cargado en 0.83 segundos
✅ Modelo: RandomForestClassifier
✅ Features: 10
✅ Clases: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

EJEMPLOS DE PREDICCIÓN RÁPIDA

📋 Paciente 1:

🏥 Nivel de Triaje: Nivel 2 - Nivel 2 - Emergencia
📊 Probabilidad: 69.00%

📋 Paciente 2:

🏥 Nivel de Triaje: Nivel 5 - Nivel 5 - No urgente
📊 Probabilidad: 91.00%

📋 Paciente 3:

🏥 Nivel de Triaje: Nivel 1 - Nivel 1 - Resucitación
📊 Probabilidad: 99.00%

PREDICCIÓN EN LOTE (30 pacientes)
✅ 6 pacientes listos para predicción
✅ Predicción completada

MEDICIÓN DE RENDIMIENTO
⏱️  100 predicciones en 3.4978 segundos
⏱️  Promedio por predicción: 0.034978 segundos

RESUMEN FINAL
✅ Modelo cargado y listo para uso
✅ Funciones disponibles:
   - predecir_triaje(): predicción individual
   - predecir_rapido(): sin gráficos (más rápido)
   - predecir_lote_rapido(): predicción en lote
   - interfaz_